In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# LLM (OpenRouter)
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("ROUTER_API_TOKEN"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0.5
)
# PROMPT + CHAIN
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert AI assistant. Use context carefully. If not found, say you don't know."
    ),
    (
        "human",
        "Context:\n{context}\n\nQuestion:\n{question}"
    )
])

parser = StrOutputParser()
basic_chain = rag_prompt | llm | parser
# LOAD PDFS
import os
from langchain_community.document_loaders import PyPDFLoader

pdf_folder = "data"
all_docs = []

for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        path = os.path.join(pdf_folder, file)
        print(f"Loading: {file}")

        loader = PyPDFLoader(path)
        docs = loader.load()
        all_docs.extend(docs)

print("Total docs:", len(all_docs))
# CHUNKING (FIXED)
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(all_docs) 

print("Chunks:", len(chunks))

# METADATA
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
    chunk.metadata["source_type"] = "pdf"
# EMBEDDINGS (FIXED OPENROUTER)
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    api_key=os.getenv("ROUTER_API_TOKEN"),
    base_url="https://openrouter.ai/api/v1",
    model="text-embedding-3-small"
)

# VECTOR DB
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embedding_model)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
# BASIC RAG
question = "What is the document about?"

retrieved_docs = retriever.invoke(question)

context = "\n\n".join([d.page_content for d in retrieved_docs])

response = basic_chain.invoke({
    "context": context,
    "question": question
})

print("\n===== BASIC RAG =====")
print(response)
# QUERY REWRITE (ADVANCED RAG)
rewrite_prompt = ChatPromptTemplate.from_template(
    "Rewrite this question for better retrieval:\n{question}"
)

rewrite_chain = rewrite_prompt | llm | parser

rewritten_query = rewrite_chain.invoke({"question": question})

print("\n===== REWRITTEN QUERY =====")
print(rewritten_query)

advanced_docs = retriever.invoke(rewritten_query)

advanced_context = "\n\n".join([d.page_content for d in advanced_docs])

advanced_answer = basic_chain.invoke({
    "context": advanced_context,
    "question": question
})

print("\n===== ADVANCED RAG =====")
print(advanced_answer)
# TOOLS (FIXED DOCSTRING ISSUE)
from langchain_core.tools import tool

@tool
def calculator(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

@tool
def retrieve_documents(query: str) -> str:
    """Retrieve documents from vector database."""
    docs = retriever.invoke(query)
    return "\n\n".join([d.page_content for d in docs])

tools = [calculator, multiply, retrieve_documents]

# LANGGRAPH MEMORY
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
# LANGGRAPH AGENT
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode

llm_with_tools = llm.bind_tools(tools)

def assistant(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: MessagesState):
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "tools"
    return END

builder = StateGraph(MessagesState)

builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

graph = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "user_1"}}
# REACT AGENT TEST
result = graph.invoke(
    {
        "messages": [
            ("user", "What is 25 multiplied by 8?")
        ]
    },
    config=config
)
print("\n===== REACT AGENT =====")
print(result["messages"][-1].content)
# AGENTIC RAG
agentic_result = graph.invoke(
    {
        "messages": [
            ("user", "Search documents and explain AI concepts")
        ]
    },
    config=config
)

print("\n===== AGENTIC RAG =====")
print(agentic_result["messages"][-1].content)

# MULTI AGENT
research_chain = ChatPromptTemplate.from_template(
    "Research specialist: {question}"
) | llm | parser

coding_chain = ChatPromptTemplate.from_template(
    "Coding specialist: {question}"
) | llm | parser

def router(q: str):
    if "code" in q.lower():
        return "coding"
    return "research"

query = "Explain vector databases"

route = router(query)

output = coding_chain.invoke({"question": query}) if route == "coding" else research_chain.invoke({"question": query})

print("\n===== MULTI AGENT =====")
print(output)

# STREAMING
print("\n===== STREAMING =====")
for chunk in llm.stream("Explain embeddings"):
    print(chunk.content, end="", flush=True)

Loading: attention.pdf
Total docs: 15
Chunks: 52

===== BASIC RAG =====
The document appears to discuss the mechanisms of attention in neural networks, particularly in the context of language processing tasks. It provides examples of how attention heads in a specific layer of a model (Layer 5 of 6) are involved in tasks such as anaphora resolution and handling long-distance dependencies in sentences. The text includes visualizations and analysis of attention patterns, indicating how different heads focus on various aspects of sentence structure and meaning. Additionally, it references several academic works related to neural machine translation and natural language processing.

===== REWRITTEN QUERY =====
What topics or themes are covered in the document?

===== ADVANCED RAG =====
The document discusses attention mechanisms in neural network architectures, specifically focusing on how these mechanisms handle long-distance dependencies and anaphora resolution in text. It provides exampl